# Begin

In [1]:
# @launchit.collected

In [2]:
import os # @launchit.collect
import sys # @launchit.collect
import copy
from collections import namedtuple, defaultdict, Counter # @launchit.collect
import itertools
import json
import datetime
import pprint
from functools import cache
import re
import pickle
from unittest.mock import Mock
import dataclasses # @launchit.collect
from dataclasses import dataclass # @launchit.collect
import IPython
from enum import Flag, StrEnum, auto # @launchit.collect
import multiprocessing as mp

from tqdm.notebook import tqdm

import lark # @launchit.collect

import numpy as np
import einops
import matplotlib.pyplot as plt
import scipy.signal as signal
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import KFold

import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch.utils.data import DataLoader, StackDataset
from torchvision import datasets

import optuna 
from optuna.storages import JournalStorage 
from optuna.storages.journal import JournalFileBackend 
from optuna.trial import TrialState

project_root_path = '${PROJECT_ROOT_PATH}' # @launchit.collect
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append('.')
from subproject import *
sys.path.append(os.path.join(project_root_path, 'lib')) # @launchit.collect
from utils import * # @launchit.collect
from logging_utils import *
from model_registry import *
import launchit # @launchit.disable
import optuna_multiprocessing  # @launchit.collect_2
from metrics_collector import RmqSummaryWriter
from autoincrement import Autoincrement
from torch_helpers import *

# Setup

In [3]:
LOG = Logging.get()
RNG = np.random.default_rng()
METRICS_SUITE = defaultdict(list)

class ExecMode(StrEnum):
    MASTER_NOTEBOOK = auto()
    LAUNCH_NOTEBOOK = auto()
    LAUNCH_MODULE = auto()

CONFIG = namedtuple('Config', 
                    'project_root_path, project_root_uri, subproject_path, data_path, private_data_path, run_path, ' + 
                    'self_fname, self_name, ' +
                    'subproject_name,' +
                    'is_cuda, cuda_device, exec_mode, is_interactive')(
    project_root_path=project_root_path,
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    subproject_path=os.path.abspath('.'),
    data_path=os.path.join(project_root_path, 'data'),
    private_data_path=None,
    run_path='',
    self_fname='',
    self_name='',
    subproject_name='',
    is_cuda=torch.cuda.is_available(),
    cuda_device='cuda' if torch.cuda.is_available() else 'cpu',
    exec_mode=ExecMode.MASTER_NOTEBOOK,
    is_interactive=True,
)

if IPython.get_ipython() is None:
    module_fname = __file__
    module_basename = os.path.basename(module_fname)
    module_name, _ = os.path.splitext(module_basename)
    
    CONFIG = CONFIG._replace(self_fname=module_fname, self_name=module_name)
    CONFIG = CONFIG._replace(exec_mode=ExecMode.LAUNCH_MODULE)
else:
    with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
        notebook_fname = json.load(cf)['jupyter_session']
        notebook_basename = os.path.basename(notebook_fname)
        notebook_name, notebook_ext = os.path.splitext(notebook_basename)
    
        m = re.match(r'(\w+)-Copy\d+$', notebook_name)
    
        if m: notebook_name = m.group(1) # e.g. Cuml is used to be launched from the copy of the notebook

        CONFIG = CONFIG._replace(self_fname=notebook_fname, self_name=notebook_name)
        
        is_launch = re.match(r'\w+-launch\d+$', notebook_name) is not None
        CONFIG = CONFIG._replace(exec_mode=ExecMode.MASTER_NOTEBOOK if not is_launch else ExecMode.LAUNCH_NOTEBOOK)

CONFIG = CONFIG._replace(is_interactive=CONFIG.exec_mode != ExecMode.LAUNCH_MODULE)

LOG.app_name = CONFIG.self_name
LOG.enable('syslog', not CONFIG.is_interactive)
LOG.enable('stdout', CONFIG.is_interactive)

CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.self_fname)))
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG = CONFIG._replace(private_data_path=os.path.join(CONFIG.data_path, CONFIG.subproject_name))
LOG(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n', when=CONFIG.is_interactive)
LOG(f'CONFIG={CONFIG._asdict()}', when=not CONFIG.is_interactive)

os.makedirs(CONFIG.private_data_path, exist_ok=True)
os.makedirs(CONFIG.run_path, exist_ok=True)

CONFIG=
{'project_root_path': '/home/misha/dev/mine/neurovision',
 'project_root_uri': 'com.develorium.neurovision',
 'subproject_path': '/home/misha/dev/mine/neurovision/12_rnn',
 'data_path': '/home/misha/dev/mine/neurovision/data',
 'private_data_path': '/home/misha/dev/mine/neurovision/data/12_rnn',
 'run_path': '/home/misha/dev/mine/neurovision/run/12_rnn',
 'self_fname': '/home/misha/dev/mine/neurovision/12_rnn/gen_text_rnn_01.ipynb',
 'self_name': 'gen_text_rnn_01',
 'subproject_name': '12_rnn',
 'is_cuda': True,
 'cuda_device': 'cuda',
 'exec_mode': <ExecMode.MASTER_NOTEBOOK: 'master_notebook'>,
 'is_interactive': True}



In [4]:
# @launchit.disable
# @launchit.collect
class LaunchGoal(StrEnum):
    UNSPECIFIED = auto()
    TRAIN_MODEL = auto()

class LaunchGoalObj(namedtuple('LaunchGoalObj', 'goal, model_group_uri, model_name, model_version, model_main_asset_fname')):
    def matches(self, *g):
        if CONFIG.exec_mode == ExecMode.MASTER_NOTEBOOK:
            return True
            
        if g and isinstance(g[0], list):
            assert len(g) == 1
            return self.goal in g[0]

        return self.goal in g

LAUNCH_GOAL = LaunchGoalObj(
    goal=LangUtils.from_str(LaunchGoal, '${LAUNCH_GOAL}', LaunchGoal.UNSPECIFIED),
    model_group_uri='${MODEL_GROUP_URI}',
    model_name='${MODEL_NAME}',
    model_version=LangUtils.from_str(int, '${MODEL_VERSION}', 0),
    model_main_asset_fname='${LAUNCHIT_FNAME}',
)
# @launchit.stop

LAUNCH_GOAL = LAUNCH_GOAL._replace(goal=LaunchGoal.UNSPECIFIED)
LAUNCH_GOAL = LAUNCH_GOAL._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
LAUNCH_GOAL = LAUNCH_GOAL._replace(model_name=CONFIG.self_name)
LAUNCH_GOAL = LAUNCH_GOAL._replace(model_version=0)
LAUNCH_GOAL = LAUNCH_GOAL._replace(model_main_asset_fname=CONFIG.self_fname)
# @launchit.stop

LOG(f'LAUNCH_GOAL=\n{pprint.pformat(LAUNCH_GOAL._asdict(), sort_dicts=False)}', when=CONFIG.is_interactive)
LOG(f'LAUNCH_GOAL={LAUNCH_GOAL._asdict()}', when=not CONFIG.is_interactive)

LAUNCH_GOAL=
{'goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'model_group_uri': 'com.develorium.neurovision.12_rnn',
 'model_name': 'gen_text_rnn_01',
 'model_version': 0,
 'model_main_asset_fname': '/home/misha/dev/mine/neurovision/12_rnn/gen_text_rnn_01.ipynb'}


# Hyperparameters

In [5]:
# @launchit.disable
# @launchit.collect
@dataclass
class Hyperparameters:
    random_seed: int = None
    # input data params
    db_fname: str = None
    # model structure params
    embedding_size: int = None
    state_model_units: list[str] = None 
    lr_model_units: list[str] = None
    # training params
    batch_size: int = None
    epochs_count: int = None
    optimizer: str = None
    learn_rate: float | str = None

    @staticmethod
    def from_dict(d):
        hp = Hyperparameters(**d)
        return hp

    def _asdict(self):
        return dataclasses.asdict(self)

HP = Hyperparameters()
HP.random_seed = 42


In [6]:
# grammar = '''
#     spec: lstm_spec # | rnn_spec | gru_spec

#     lstm_spec: "LSTM" "(" LSTM_HIDDEN_SIZE ")" ("x" LSTM_NUM_LAYERS)?
#     LSTM_HIDDEN_SIZE: NUMBER
#     LSTM_NUM_LAYERS: NUMBER
    
#     %import common.WORD
#     %import common.NUMBER
#     %import common.WS
#     %ignore WS
# '''
# parser = lark.Lark(grammar, start='spec')
# print(parser.parse('LSTM(64)x2').pretty())

In [7]:
@dataclass
class StateModelUnitParams: 
    module: str = None
    hidden_size: int = None
    num_layers: int = None

@dataclass
class LogRegModelUnitParams: 
    items_count: int = None
    nonlinearity: str = None # any of: None, sigmoid, tanh, ...
    dropout: float = None

@dataclass
class LearnRateParams: 
    Plateau = namedtuple('Plateau', 'factor patience')
    
    learn_rate: float = None
    plateau: object = None

def get_lark_tree_value(tree, var_name, default_value=None):
    try:
        return next(tree.scan_values(lambda i: i.type == var_name)).value
    except StopIteration:
        return default_value

def hp_parse_state_model_units(self):
    grammar = '''
        spec: lstm_spec # | rnn_spec | gru_spec
    
        lstm_spec: "LSTM" "(" LSTM_HIDDEN_SIZE ")" ("x" LSTM_NUM_LAYERS)?
        LSTM_HIDDEN_SIZE: NUMBER
        LSTM_NUM_LAYERS: NUMBER
        
        %import common.WORD
        %import common.NUMBER
        %import common.WS
        %ignore WS
    '''
    parser = lark.Lark(grammar, start='spec')
    params_list = []

    for unit in self.state_model_units:
        tree = parser.parse(unit)
        gtv = lambda var_name, default_value='': get_lark_tree_value(tree, var_name, default_value)
        params = StateModelUnitParams()

        if list(tree.find_data('lstm_spec')):
            params.module = 'LSTM'
            params.hidden_size = int(gtv('LSTM_HIDDEN_SIZE'))
            params.num_layers = int(gtv('LSTM_NUM_LAYERS', 1))
            
        params_list.append(params)

    return params_list

def hp_parse_lr_model_units(self):
    grammar = '''
        spec: (dropout_spec "->")? items_count_spec ("->" nonlinearity_spec)?
    
        dropout_spec: "dropout" "(" DROPOUT_P ")"
        DROPOUT_P: NUMBER
    
        items_count_spec: ITEMS_COUNT
        ITEMS_COUNT: NUMBER
    
        nonlinearity_spec: NONLINEARITY_WORD
        NONLINEARITY_WORD: WORD
    
        %import common.WORD
        %import common.NUMBER
        %import common.WS
        %ignore WS
    '''
    parser = lark.Lark(grammar, start='spec')
    params_list = []

    for unit in self.lr_model_units:
        tree = parser.parse(unit)
        gtv = lambda var_name, default_value='': get_lark_tree_value(tree, var_name, default_value)
        params = LogRegModelUnitParams()
        params.items_count = int(gtv('ITEMS_COUNT'))
        params.nonlinearity = gtv('NONLINEARITY_WORD')
        params.dropout = float(gtv('DROPOUT_P')) if gtv('DROPOUT_P', False) else None
        params_list.append(params)

    return params_list

def hp_parse_learn_rate(self):
    params = LearnRateParams()

    if isinstance(self.learn_rate, float):
        params.learn_rate = self.learn_rate
        return params
        
    grammar = '''
        spec: initial_lr_spec(","plateau_spec)?
    
        initial_lr_spec: INITIAL_LR
        INITIAL_LR: NUMBER
    
        plateau_spec: "plateau" "(" (|plateau_params_spec ("," plateau_params_spec)*) ")"
        plateau_params_spec: plateau_factor_spec | plateau_patience_spec
        plateau_factor_spec: "factor" "=" plateau_factor_value_spec
        plateau_factor_value_spec: PLATEAU_FACTOR_VALUE
        plateau_patience_spec: "patience" "=" plateau_patience_value_spec
        plateau_patience_value_spec: PLATEAU_PATIENCE_VALUE
        PLATEAU_FACTOR_VALUE: NUMBER
        PLATEAU_PATIENCE_VALUE: NUMBER
        
        %import common.NUMBER
        %import common.WS
        %ignore WS
    '''
    parser = lark.Lark(grammar, start='spec')
    tree = parser.parse(self.learn_rate)
    gtv = lambda var_name, default_value='': get_lark_tree_value(tree, var_name, default_value)
    params.learn_rate = float(gtv('INITIAL_LR'))
    
    if list(tree.find_data('plateau_spec')):
        params.plateau = LearnRateParams.Plateau(
            factor=float(gtv('PLATEAU_FACTOR_VALUE', 0.1)),
            patience=int(gtv('PLATEAU_PATIENCE_VALUE', 10)),
        )

    return params

Hyperparameters.parse_state_model_units = hp_parse_state_model_units
Hyperparameters.parse_lr_model_units = hp_parse_lr_model_units
Hyperparameters.parse_learn_rate = hp_parse_learn_rate

In [8]:
# @launchit.disable
x = Hyperparameters(state_model_units=['LSTM(64)x10', 'LSTM(32)'])
x.parse_state_model_units()

[StateModelUnitParams(module='LSTM', hidden_size=64, num_layers=10),
 StateModelUnitParams(module='LSTM', hidden_size=32, num_layers=1)]

In [9]:
# @launchit.disable
x = Hyperparameters(lr_model_units=[
    'dropout(0.5)->200->tanh',
    '200->sigmoid',
])
x.parse_lr_model_units()

[LogRegModelUnitParams(items_count=200, nonlinearity='tanh', dropout=0.5),
 LogRegModelUnitParams(items_count=200, nonlinearity='sigmoid', dropout=None)]

In [10]:
# @launchit.disable
x = Hyperparameters(learn_rate='0.005,plateau(factor=0.115, patience=15)')
lr_params = x.parse_learn_rate()
o = torch.optim.Adam(params=[nn.Parameter(torch.tensor(1.))])
p = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=o, **lr_params.plateau._asdict())
lr_params, p.factor, p.patience

(LearnRateParams(learn_rate=0.005, plateau=Plateau(factor=0.115, patience=15)),
 0.115,
 15)

# Launch

## new_model_registry

In [11]:
def new_model_registry(is_real=None, group_uri=None):
    is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
    is_real = is_real if is_real is not None else is_launch

    if not is_real:
        mr = Mock()
        mr.register_model.return_value = 0
        return mr
        
    return ModelRegistry(LAUNCH_GOAL.model_group_uri if group_uri is None else group_uri)

## new_summary_writer

In [12]:
def new_summary_writer(log_dir, is_real=None):
    is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
    is_real = is_real if is_real is not None else is_launch

    if not is_real:
        sw = Mock()
        sw.flush.side_effect = sw.reset_mock # to get rid of all recorded call_args_list, which might be heavy (e.g. add_figure)
        return sw
    
    return RmqSummaryWriter(log_dir)

## Bootstrap

In [13]:
optuna_trial = optuna_multiprocessing.get_trial()
optuna_trial_subdir_name = ''

if optuna_trial is not None:
    optuna_trial.set_user_attr('MODEL_VERSION', LAUNCH_GOAL.model_version)
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    optuna_trial_subdir_name = f'opt_{study_serial}'
    LOG(f'Optuna {optuna_trial.number=}, {optuna_trial.user_attrs=}')

LOG(f'HP={HP._asdict()}', when=not CONFIG.is_interactive)
    
if HP.random_seed is not None:
    torch.manual_seed(HP.random_seed)
    RNG = np.random.default_rng(HP.random_seed)    
    LOG(f'Random seed={HP.random_seed}')

if True or LAUNCH_GOAL.goal != LaunchGoal.TRAIN_MODEL_AFTER_CV:
    model_registry = new_model_registry()
    model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, CONFIG.self_fname, replace=True)
        
    meta = dict(
        optuna_trial_number=getattr(optuna_trial, 'number', None),
        hypers=HP._asdict(), 
        config=CONFIG._asdict(), 
    )
    
    with io.StringIO() as b:
        json.dump(meta, b)
        model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, b, asset_ext='json', asset_classifier='meta', replace=True)

summary_log_dir = LAUNCH_GOAL.model_name
summary_log_dir = os.path.join(summary_log_dir, optuna_trial_subdir_name) if optuna_trial_subdir_name != '' else summary_log_dir 
summary_log_dir = os.path.join(summary_log_dir, str(LAUNCH_GOAL.model_version))
LOG(f'Tensorboard run={summary_log_dir}')
summary_writer = new_summary_writer(log_dir=summary_log_dir)
summary_writer.add_text('hypers', pprint.pformat(HP._asdict(), sort_dicts=False), 1)
summary_writer.add_text('config', pprint.pformat(CONFIG._asdict(), sort_dicts=False), 1)

Random seed=42
Tensorboard run=gen_text_rnn_01/0


<Mock name='mock.add_text()' id='140468271359296'>

In [14]:
mr = new_model_registry(is_real=True, group_uri='org.gutenberg')
# mr.get_asset_content('ulysses', 1, 'txt')
# mr.get_assets('The*Great*Gatsby', 1)

# Model

## StateModelUnit

In [15]:
class StateModelUnit(nn.Module):
    def __init__(self, input_size, unit_hp):
        super().__init__()

        match unit_hp.module:
            case 'LSTM':
                self.module = nn.LSTM(input_size, unit_hp.hidden_size, num_layers=unit_hp.num_layers, batch_first=True)
            case _:
                assert False, f'Unsupported {unit_hp.module=}'

        self.output_size = self.module.hidden_size

    def forward(self, inp): 
        # inp.shape: batch, chunk, input (of embedding_size or of ouput_size of preceeding unit)
        res, (hidden, cell) = self.module(inp)
        return res

## StateModel

In [16]:
class StateModel(nn.Module):
    def __init__(self, input_size, model_hp):
        super().__init__()
        self.units = nn.Sequential()

        for unit_hp in model_hp.parse_state_model_units():
            unit = StateModelUnit(input_size, unit_hp)
            self.units.append(unit)
            input_size = unit.output_size

        self.output_size = self.units[-1].output_size

    def forward(self, inp):
        return self.units(inp)

In [17]:
# @launchit.disable
embedding_size = 128
model_hp = Hyperparameters(state_model_units=[
    'LSTM(64)x5',
    'LSTM(32)x3'
])
model = StateModel(embedding_size, model_hp)
probe_tensor = torch.ones(1, 10, embedding_size) # 1 sample of chunk of 10-elements each element consisting of embedding_size
res = model(probe_tensor)
assert res.shape[-1] == model.output_size
print(model)
print(f'Parameters count={sum([p.numel() for p in model.parameters()]):}')
print(f'Output shape={res.shape}')

StateModel(
  (units): Sequential(
    (0): StateModelUnit(
      (module): LSTM(128, 64, num_layers=5, batch_first=True)
    )
    (1): StateModelUnit(
      (module): LSTM(64, 32, num_layers=3, batch_first=True)
    )
  )
)
Parameters count=212224
Output shape=torch.Size([1, 10, 32])


## LogRegModelUnit

In [18]:
class LogRegModelUnit(nn.Module):
    def __init__(self, input_dims_count, unit_hp):
        super().__init__()
        self.linear = nn.Linear(
            in_features=input_dims_count, 
            out_features=unit_hp.items_count, 
            bias=True
        )

        if unit_hp.nonlinearity is None:
            self.nonlinearity = lambda i: i
        else:
            self.nonlinearity = getattr(F, unit_hp.nonlinearity)

        if unit_hp.dropout is None:
            self.dropout = lambda i: i
        else:
            assert unit_hp.dropout >= 0
            self.dropout = nn.Dropout(unit_hp.dropout)

        self.output_size = self.linear.out_features

    def forward(self, inp): 
        res = self.dropout(inp)
        res = self.linear(res)
        res = self.nonlinearity(res)
        return res

## LogRegModel

In [19]:
class LogRegModel(nn.Module):
    def __init__(self, input_size, model_hp):
        super().__init__()
        self.units = nn.Sequential()

        for unit_hp in model_hp.parse_lr_model_units():
            unit = LogRegModelUnit(input_size, unit_hp)
            self.units.append(unit)
            input_size = unit.output_size

        if self.units:
            self.output_size = self.units[-1].output_size
        else:
            self.output_size = input_size

    def forward(self, inp):
        if not self.units:
            return inp

        return self.units(inp)

In [20]:
# @launchit.disable
input_size = 128
model_hp = Hyperparameters(lr_model_units=[
    'dropout(0.5)->200->tanh',
    '80->sigmoid',
])
model = LogRegModel(input_size, model_hp)
probe_tensor = torch.ones(1, 10, input_size)
res = model(probe_tensor)
assert res.shape[-1] == model.output_size
print(model)
print(f'Parameters count={sum([p.numel() for p in model.parameters()]):}')
print(f'Output shape={res.shape}')

LogRegModel(
  (units): Sequential(
    (0): LogRegModelUnit(
      (linear): Linear(in_features=128, out_features=200, bias=True)
      (dropout): Dropout(p=0.5, inplace=False)
    )
    (1): LogRegModelUnit(
      (linear): Linear(in_features=200, out_features=80, bias=True)
    )
  )
)
Parameters count=41880
Output shape=torch.Size([1, 10, 80])


## MainModel

In [21]:
class MainModel(nn.Module):
    def __init__(self, vocab_size, model_hp):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, model_hp.embedding_size)
        self.state_model = StateModel(model_hp.embedding_size, model_hp)
        self.lr_model = LogRegModel(self.state_model.output_size, model_hp)
        self.output_layer = nn.Linear(self.lr_model.output_size, vocab_size)
        self.output_size = vocab_size

    def forward(self, inp):
        res = self.embedding(inp)
        res = self.state_model(res)
        res = res[:,-1,:] # take features from last step only (-1)
        res = self.lr_model(res)
        res = self.output_layer(res)
        return res

In [22]:
# @launchit.disable
vocab_size = 10_000
model_hp = Hyperparameters(
    embedding_size=256,
    state_model_units=[
        'LSTM(64)x5',
        'LSTM(32)x3',
        ],
    lr_model_units=[
        'dropout(0.5)->200->tanh',
        '80->sigmoid',
    ],
)
model = MainModel(vocab_size, model_hp)
probe_tensor = torch.ones(2, 10, dtype=int) # batch of 2 sample each containing 10 sequences (chunks, eachk elem of chunk is a token)
res = model(probe_tensor)
assert res.shape[-1] == model.output_size
sum([p.numel() for p in model.parameters()])
print(model)
print(f'Parameters count={sum([p.numel() for p in model.parameters()]):}')
print(f'Output shape={res.shape}')

MainModel(
  (embedding): Embedding(10000, 256)
  (state_model): StateModel(
    (units): Sequential(
      (0): StateModelUnit(
        (module): LSTM(256, 64, num_layers=5, batch_first=True)
      )
      (1): StateModelUnit(
        (module): LSTM(64, 32, num_layers=3, batch_first=True)
      )
    )
  )
  (lr_model): LogRegModel(
    (units): Sequential(
      (0): LogRegModelUnit(
        (linear): Linear(in_features=32, out_features=200, bias=True)
        (dropout): Dropout(p=0.5, inplace=False)
      )
      (1): LogRegModelUnit(
        (linear): Linear(in_features=200, out_features=80, bias=True)
      )
    )
  )
  (output_layer): Linear(in_features=80, out_features=10000, bias=True)
)
Parameters count=3637672
Output shape=torch.Size([2, 10000])


# Model training

## Configure

In [54]:
# @launchit.disable
# @launchit.collect_1
HP.db_fname = 'bundle_40_symbol.db'
HP.embedding_size = 256
HP.state_model_units = [
    'LSTM(512)'
]
HP.lr_model_units = [
]
HP.batch_size = 100
HP.epochs_count = 1
HP.optimizer = 'Adam'
HP.learn_rate = '0.005,plateau()'
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'random_seed': 42,
 'db_fname': 'bundle_40_symbol.db',
 'embedding_size': 256,
 'state_model_units': ['LSTM(512)'],
 'lr_model_units': [],
 'batch_size': 100,
 'epochs_count': 1,
 'optimizer': 'Adam',
 'learn_rate': '0.005,plateau()'}


## Create

In [55]:
dataset = ChunkDataset(os.path.join(CONFIG.private_data_path, HP.db_fname), prefetch_buffer_size=HP.batch_size, rng=RNG)
data_loader = DataLoader(dataset, batch_size=HP.batch_size)
model = MainModel(dataset.vocab_size, HP)
model = model.to(device=CONFIG.cuda_device)
loss_fn = nn.CrossEntropyLoss()
lr_params = HP.parse_learn_rate()
optimizer = getattr(torch.optim, HP.optimizer)(model.parameters(), lr=lr_params.learn_rate)
lr_scheduler = LrSchedulerWrapper(optimizer, lr_params)

## Train

In [ ]:
# @launchit.disable_2
for epoch in tqdm(range(1, HP.epochs_count + 1), 'Epoch', disable=not CONFIG.is_interactive):
    epoch_loss_sum, epoch_loss_denom = 0, 0
    
    for batch in tqdm(data_loader, 'Batch', disable=not CONFIG.is_interactive, leave=False):
        optimizer.zero_grad()
        batch = batch.to(device=CONFIG.cuda_device)
        input_chunk = batch[:,:-1] # all but last column
        target_tokens = batch[:,-1] # last column
        
        logits = model(input_chunk)
        loss = loss_fn(logits, target_tokens)
        loss.backward()
        
        optimizer.step()
        
        epoch_loss_sum += loss.item() * len(batch)
        epoch_loss_denom += len(batch)

    assert epoch_loss_denom > 0
    train_loss = epoch_loss_sum / epoch_loss_denom
    summary_writer.add_scalar('train_loss', train_loss, epoch - 1)
    METRICS_SUITE['train_loss'].append(train_loss)

    lr_scheduler.step(train_loss)
    summary_writer.flush()

## Save

In [80]:
# @launchit.disable_2
model_registry = new_model_registry()

with io.BytesIO() as b:
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'hypers': HP._asdict(),
    }, b)
    model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, b, asset_ext='pt', asset_classifier='main', replace=True)

with io.StringIO() as b:
    json.dump(METRICS_SUITE, b)
    model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, b, asset_ext='json', asset_classifier='metrics', replace=True)

# Save Optuna trial result

In [121]:
# @launchit.disable_1
optuna_trial = optuna_multiprocessing.get_trial()

if optuna_trial is not None:
    if 'IS_PRUNED' in optuna_trial.user_attrs:
        raise optuna.exceptions.TrialPruned()

    if not METRICS_SUITE:
        LOG(f'Empty metrics suite. Cancelling model')
        optuna_multiprocessing.save_trial_result(0)
    else:
        match LAUNCH_GOAL.goal:
            case LaunchGoal.TRAIN_MODEL_CV:
                last_std_val_accuracy = METRICS_SUITE['std_val_accuracy'][-1]
                last_mean_val_accuracy = METRICS_SUITE['mean_val_accuracy'][-1]
                    
                if last_std_val_accuracy > 0.05:
                    LOG(f'Unstable condition encountered: {last_mean_val_accuracy=}, {last_std_val_accuracy=}. Cancelling model')
                    optuna_multiprocessing.save_trial_result(0)
                else:
                    optuna_multiprocessing.save_trial_result(last_mean_val_accuracy)
                    LOG(f'Train objective result: {last_mean_val_accuracy=}')
            case LaunchGoal.TRAIN_MODEL:
                # Bad practice due to test data influences model selection
                last_test_accuracy = METRICS_SUITE['test_accuracy'][-1]
                optuna_multiprocessing.save_trial_result(last_test_accuracy)
                LOG(f'Train objective result: {last_test_accuracy=}')
            case _:
                assert False, f'Unsupported {LAUNCH_GOAL.goal=}'

# Launch creation

## TRAIN_MODEL

In [23]:
# @launchit.disable
launchit_t0 = time.time()

In [24]:
# @launchit.disable
launchit_interval = time.time() - launchit_t0

if launchit_interval > 0.05:
    model_version = int(Autoincrement.get(f'{LAUNCH_GOAL.model_group_uri}.{LAUNCH_GOAL.model_name}'))
    assert model_version > 0, model_version
    model_registry_obj = new_model_registry(is_real=True)
    model_registry_obj.register_model(LAUNCH_GOAL.model_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=LAUNCH_GOAL.model_group_uri,
        MODEL_NAME=LAUNCH_GOAL.model_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL=LaunchGoal.TRAIN_MODEL,
    )
    launch_notebook_fname = launchit.launchit(CONFIG.self_fname, launch_serial=model_version, expandvars=expandvars, collect_inds=[1], disable_inds=[])
    LOG(f'Created launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

Model instance registered, version=1
Creating /home/misha/dev/mine/neurovision/12_rnn/gen_text_rnn_01-launch1.ipynb
Created launch notebook "/home/misha/dev/mine/neurovision/12_rnn/gen_text_rnn_01-launch1.ipynb"


## Optuna (model selection)

### Templates

In [28]:
# @launchit.disable
# @launchit.collect_2
optuna_trial = optuna_multiprocessing.get_trial()

if optuna_trial is not None:
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    
    match study_serial:
        # case 1:
        #     HP.random_seed = 42
        #     HP.images_preprocessing = 'SAMPLE_WISE' # NONE, UNINORM, SAMPLE_WISE, MIN_MAX, STANDARDIZE, ZCA_WHITEN, ZCA_HFR30_WHITEN
        #     HP.fe_model_units = [
        #         'conv(1->32(1)x5+bias)->tanh->gain->abs->avg(2)',
        #         'conv(32->64(16)x5+bias)->tanh->gain->abs->avg(2)'
        #     ]
        #     HP.lr_model_units = [
        #         '200->tanh',
        #         '10->tanh',
        #     ]
        #     HP.refine_fe_model = True
        #     HP.batch_size = 100
        #     HP.epochs_count = 15
        #     HP.optimizer = 'Adam'
        #     initial_lr = optuna_trial.suggest_float('initial_lr', 0.0005, 0.005)
        #     plateau_factor = optuna_trial.suggest_float('plateau_factor', 0.1, 0.5)
        #     plateau_patience = optuna_trial.suggest_int('plateau_patience', HP.epochs_count // 4, HP.epochs_count // 2)
        #     HP.learn_rate = f'{initial_lr}, plateau(factor={plateau_factor}, patience={plateau_patience})'
        #     HP.cv_folds_count = 5
        case _:
            assert False, f'Unsupported {study_serial=}'    

### Unleash

In [ ]:
# @launchit.disable
def get_optimize_directions(lg):
    match lg:
        case LaunchGoal.TRAIN_MODEL_CV:
            return ['maximize']
        case LaunchGoal.TRAIN_MODEL:
            return ['maximize']
        case _:
            assert False, f'Unsupported {lg=}'

lg = LaunchGoal.TRAIN_MODEL_CV
expandvars = dict(
    PROJECT_ROOT_PATH=CONFIG.project_root_path,
    MODEL_GROUP_URI=LAUNCH_GOAL.model_group_uri,
    MODEL_NAME=LAUNCH_GOAL.model_name,
    LAUNCH_GOAL=lg.value,
)
study_serial = 3
study_name = f'{CONFIG.self_name}_{expandvars['LAUNCH_GOAL']}_{study_serial}'
rop_task = optuna_multiprocessing.RunOptimizationTask(
    app_name=CONFIG.self_name,
    is_stdout_enabled=False,
    notebook_fname=CONFIG.self_fname,
    notebook_name=CONFIG.self_name,
    model_group_uri=LAUNCH_GOAL.model_group_uri,
    model_name=LAUNCH_GOAL.model_name,
    expandvars=expandvars,
    collect_inds=[2],
    disable_inds=[2],
    run_path=CONFIG.run_path,
    study_serial=study_serial,
    study_name=study_name,
    study_fname=os.path.join(CONFIG.run_path, study_name + '.log'),
    optimize_directions=get_optimize_directions(lg),
)
rop_tasks = [rop_task] * 25
mp_ctx = mp.get_context('spawn') # Req-d for CUDA, fork doesn't work within PyTorch

with mp_ctx.Pool(processes=2, maxtasksperchild=1) as pool:  # maxtasksperchild=1 forces fresh process for each trial to spare resources and avoid possible side effects of processe resue
    pool.map(optuna_multiprocessing.run_optimization, rop_tasks)

### Stats

In [ ]:
# @launchit.disable
study = optuna.create_study(
    study_name=rop_task.study_name,
    storage=JournalStorage(JournalFileBackend(file_path=rop_task.study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs['MODEL_VERSION']}')
    
    LOG('  Params: ')
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    print(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        print(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        print(f"\tnumber: {trial.number}")
        print(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        print(f"\tparams: {trial.params}")
        print(f"\tvalues: {trial.values}")